In [1]:
from langchain_core.documents import Document

In [2]:
sample_doc = Document(
    page_content="Hello World!",
    metadata = {"source":"https://google.com"}

)

In [3]:
sample_doc

Document(metadata={'source': 'https://google.com'}, page_content='Hello World!')

In [4]:
from langchain_community.document_loaders.text import TextLoader

C:\Users\Asus\AppData\Local\Temp\ipykernel_8392\3315532343.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [5]:
loader = TextLoader("data/Python.txt",encoding="utf-8")

In [6]:
doc = loader.load()

In [7]:
doc

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a widely used, high-level programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming, making it versatile for different types of projects. Python is commonly used in fields such as web development, data analysis, artificial intelligence, and automation. Its extensive standard library and large community make it easy for developers to find tools and support, which contributes to its popularity among beginners and professionals alike.')]

In [8]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data/kek.pdf")

document = pdf_loader.load()

In [9]:
document

[Document(metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Executive Summary', 'author': 'ChatGPT Deep Research', 'source': 'data/kek.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Executive Summary\nWe propose a cloud‐native, batch‐oriented data/ML stack with a feature‐store and strong governance to\nimplement the AMC Renewal Retention predictor . For each layer (warehouse, ingestion, feature store, etc.)\nwe evaluate 2–3 leading open‐source or vendor options (considering cost, scale, latency, team skill) and\nnote how they integrate. We also inventory all required data sources (CRM, contracts, billing, service tickets,\ntelemetry, finance, sales logs, etc.) detailing key fields, freshness, retention, privacy concerns, and\npoint‑in‑time labeling. We design a point‑in‑time feature schema (avoiding “future leakage”), with example\nSQL/DBT snippets for joins and label creation. We outline the team roles (data engineers, 

# Loading

In [10]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdfs():
    path = "data/pdfs"
    num_docs = 0
    all_docs = []
    for file in os.listdir(path):
        if file.lower().endswith(".pdf"):
            pdf_path = os.path.join(path,file)
            pdf = PyPDFLoader(pdf_path)
            doc = pdf.load()
            all_docs.extend(doc)
            num_docs+=1
    print(f"Loaded {num_docs} PDFs || Total pages: {len(all_docs)}")
    return all_docs

In [12]:
all_pdfs = load_all_pdfs()

Loaded 2 PDFs || Total pages: 16


In [13]:
all_pdfs

[Document(metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Executive Summary', 'author': 'ChatGPT Deep Research', 'source': 'data/pdfs\\research1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Executive Summary\nWe propose a cloud‐native, batch‐oriented data/ML stack with a feature‐store and strong governance to\nimplement the AMC Renewal Retention predictor . For each layer (warehouse, ingestion, feature store, etc.)\nwe evaluate 2–3 leading open‐source or vendor options (considering cost, scale, latency, team skill) and\nnote how they integrate. We also inventory all required data sources (CRM, contracts, billing, service tickets,\ntelemetry, finance, sales logs, etc.) detailing key fields, freshness, retention, privacy concerns, and\npoint‑in‑time labeling. We design a point‑in‑time feature schema (avoiding “future leakage”), with example\nSQL/DBT snippets for joins and label creation. We outline the team roles (data

# Chunking

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size = 500,chunk_overlap = 50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = splitter.split_documents(documents=documents)
    return chunked_docs

In [15]:
chunks = split_documents(all_pdfs)

In [16]:
chunks

[Document(metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Executive Summary', 'author': 'ChatGPT Deep Research', 'source': 'data/pdfs\\research1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Executive Summary\nWe propose a cloud‐native, batch‐oriented data/ML stack with a feature‐store and strong governance to\nimplement the AMC Renewal Retention predictor . For each layer (warehouse, ingestion, feature store, etc.)\nwe evaluate 2–3 leading open‐source or vendor options (considering cost, scale, latency, team skill) and\nnote how they integrate. We also inventory all required data sources (CRM, contracts, billing, service tickets,'),
 Document(metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Executive Summary', 'author': 'ChatGPT Deep Research', 'source': 'data/pdfs\\research1.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='telemetry, finance, sales log

In [17]:
len(chunks)

78

# Embedding

In [18]:
from sentence_transformers import SentenceTransformer
from embed_manager import EmbeddingManager
from vector_store_manager import VectorStoreManager

In [19]:
vector_store = VectorStoreManager()

Vector Store Initialized @pdf_documents || Docs in collection: 0


In [20]:
texts = [doc.page_content for doc in chunks]

embedding_manager = EmbeddingManager()

embeddings = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks,embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimensions: 384
Total documents added: 78 || Total count: 78


# Retrieval Pipeline

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
from retriever import RAGRetriever

In [29]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [31]:
rag_retriever.retrieve("What skills does the person have?")

Distance: 0.7126 | Similarity: 0.2874
Distance: 0.7151 | Similarity: 0.2849
Distance: 0.7352 | Similarity: 0.2648
Distance: 0.7599 | Similarity: 0.2401
Distance: 0.7792 | Similarity: 0.2208
Retrieved 5 documents.


[{'id': 'doc_c3fecdc0-0cec-48fb-a853-1b5683fda236',
  'document': '•Churn Predictor|Mar 2026|Python, PyT orch\n–Engineered churn predictor from spending habits and services.\n–Deployed deep learning with feed-forward neural networks.\n–Optimized PyTorch model for production deployment.\nSkills\n•Languages: Python, Java, C/C++, JavaScript, TypeScript, Go\n•F rameworks: React, Next.js, FastAPI, Spring Boot, TensorFlow, PyTorch, Scikit-learn, NumPy, Pandas\n•T ools: Git, Docker, A WS, Linux, SQL, MongoDB, PostgreSQL, Redis',
  'metadata': {'doc_index': 76,
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1',
   'subject': '',
   'page': 0,
   'title': '',
   'author': '',
   'total_pages': 1,
   'moddate': '2026-03-16T10:25:06+00:00',
   'page_label': '1',
   'producer': 'pdfTeX-1.40.27',
   'source': 'data/pdfs\\resume.pdf',
   'context_length': 451,
   'trapped': '/False',
   'creationdate': 